# **GA-NLP Final Project: Customer Review Classification & Summarization**

*Image generated with Playground-v2.5 on Poe*

## **Business Context**

In today's digital era, **customer reviews are pivotal for e-commerce platforms and online service providers**, serving as a barometer for customer satisfaction, highlighting areas for improvement, and guiding business decisions. The intricate process of Aspect-based Sentiment Analysis is instrumental in dissecting these reviews, offering a granular understanding of customer sentiments towards various facets of a product or service. This nuanced approach enables businesses to pinpoint specific elements—be it the screen, keyboard, or customer service of a laptop—that may need enhancement.

Moreover, the advent of Large Language Models (LLMs) has revolutionized the way businesses approach the **summarization of customer reviews and case briefs**. By leveraging LLMs, companies can swiftly and accurately distill the essence of customer feedback, streamlining the process of understanding overall sentiments and facilitating the refinement of product offerings. This swift synthesis not only conserves time and resources but also plays a critical role in enhancing customer satisfaction, driving sales, and boosting revenue.

Additionally, this project will delve into the **fine-tuning of models to elevate the quality of summaries provided by LLMs**. This involves training models on domain-specific data to enhance their ability to generate more precise and relevant summaries, thus improving the utility of these summaries in real-world applications.

Embarking on this final project on Customer Review Aspect-based Classification & Summarization, coupled with an emphasis on model fine-tuning for enhanced summarization, equips you with invaluable skills applicable to real-world business contexts. Through hands-on experience with code and implementation specifics, you'll gain the proficiency to construct such solutions using open-source LLMs. This experience will not only serve as a compelling Proof-of-Concept but also pave the way for the deployment and productionization of these cutting-edge solutions in business environments.

## **Project Objective**

Develop a Generative AI application using a Large Language Model to **automate the Asspect-based Classification and Summarization of Customer Reviews** received by a business. The application will aim to predict review sentiment, aspect-based sentiment and generate summaries of these customer reviews, and will evaluate the performance of the LLM at these tasks through various evaluation schemes.

In [1]:
# Basic Imports for Libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import pandas as pd
import numpy as np
from tqdm import tqdm
import json
import re

import torch

from google.colab import drive
import locale

## **Question 1: Install the Necessary Libraries (4 Marks)**



In [1]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28  --force-reinstall --upgrade --no-cache-dir -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 71.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 186.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 232.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.3.0+cu121 requires nvidia-cublas-cu12==12.1.3.1; platform_system == "Linux" and platform_machine == "x86_64", which is not installed.
torch 2.3.0+cu121 requires nvidia-cuda-cupti-cu12==12.1.105; platform_system == "Linux" and platform_machine == "x86_64", which is not installed.
torch 2.3.0+cu121 requires nvidia-cuda-nvrtc-cu12==12.1.105; platform_system == "Linux" and platform_machine == "x86

In [3]:
# For downloading the models from HF Hub
!pip install huggingface_hub==0.23.2 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 5.3 MB/s eta 0:00:00


In [4]:
#@title Run this cell to setup Unsloth on Colab
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.4/102.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 19.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 24.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 18.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires numpy<2.0a0,>=1.23, but you have numpy 2.0.0 which is incompatible.
cudf-cu12 24.4.1 requi

In [5]:
#download datasets evaluate rouge_score and bert score
!pip install -q datasets==2.16.1 evaluate==0.4.1 rouge_score==0.1.2 bert_score==0.3.12


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 55.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 20.4 MB/s eta 0:00:00


In [ ]:
import evaluate

ModuleNotFoundError: No module named 'evaluate'

In [ ]:
from datasets import load_dataset

import datasets

In [ ]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 10.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.3 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.2.79-cp310-cp310-linux_x86_64.whl size=172369507 sha256=4a978f9b5c01b0c53697eb5e98750f0847117d0bfd9de43d744710a081fd2706
  Stored in directory: /root/.cache/pip/wheels/bb/2e/11/8b10c6b698e6abc1289e9919e098ac4bcf6b16ebd46153e8ba
Successfully built llama-cpp-python


In [ ]:
!pip list | grep llama

llama_cpp_python                 0.2.79


##**Question 2: Import Libraries (5 Marks)**

In [ ]:
## Import Hf_hub_download from hugging_face_hub
## Import Llama from llama_cpp
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [ ]:
# Import FastLanguageModel from unsloth
# Import SFTTrainer from trl
# Import TrainingArguments from transformers

from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

ModuleNotFoundError: No module named 'unsloth'

##**Question 3: Model Setup (4 Marks)**

HuggingFace Link: The Bloke's Llama 2 13B Chat GGUF File

https://huggingface.co/TheBloke/Llama-2-13B-GGUF/blob/main/llama-2-13b.Q5_K_M.gguf


In [ ]:
# Add Bloke Version of llama 13B Model from Hugging Face in GGUF Format Here. Also use Model Base Name Keeping in mind the 15 GB maximum GPU Ram available

model_name_or_path = "TheBloke/Llama-2-13B-GGUF"
model_basename = "llama-2-13b.Q5_K_M.gguf" # the model is in gguf format from Hugging face

In [ ]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )

In [ ]:
lcpp_llm = Llama(
        model_path=model_path,
        n_threads=2,  # CPU cores
        n_batch=512,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
        n_gpu_layers=43,  # Change this value based on your model and your GPU VRAM pool.
        n_ctx=4096,  # Context window
    )

#*Sentiment Classification*

##**Question 4: Data Preprocessing for Sentiment Classification (5 Marks)**


(A) Load a CSV File containing the Dataset of 30 Reviews (2 Marks)

(B) Divide the Dataset into Train and Test Dataset (Gold Reviews) such that Train and Test Dataset contains equal number of positive and negative reviews (3 Marks)


In [ ]:
# The data is stored on my Google Colab
# Mouting Drive and Reading the CSV
drive.mount('/content/drive')
# Load a CSV File containing Dataset of 30 Reviews
sample_reviews_df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/GenAI - 2024/Final Project/Final_Project_customer_reviews_dataset.csv")

In [ ]:
# View a sample of the data
sample_reviews_df.head(10)

In [ ]:
# Checking the values of review sentiment so we can group them by positive and negative
sample_reviews_df.review_sentiment.value_counts()

In [ ]:
# Separate positive and negative reviews
positive_reviews = sample_reviews_df.query("review_sentiment == 'Positive'")
negative_reviews = sample_reviews_df.query("review_sentiment == 'Negative'")

# Sample 5 positive and 5 negative reviews for gold examples
positive_gold_examples = positive_reviews.sample(5, random_state=40)
negative_gold_examples = negative_reviews.sample(5, random_state=40)

# Concatenate positive and negative gold examples
sample_reviews_gold_examples_df = pd.concat([positive_gold_examples, negative_gold_examples])

# Create the training set by excluding gold examples
sample_reviews_examples_df = sample_reviews_df.drop(index=sample_reviews_gold_examples_df.index)

# Convert gold examples to JSON
columns_to_select = ['review_text', 'review_sentiment']
gold_examples_json = sample_reviews_gold_examples_df[columns_to_select].to_json(orient='records')

# Print the first record from the JSON
print(json.loads(gold_examples_json)[0])

# Print the shapes of the datasets
print("Training Set Shape:", sample_reviews_examples_df.shape)
print("Gold Examples Shape:", sample_reviews_gold_examples_df.shape)


In [ ]:
# Viewing the sample reviews gold examples
sample_reviews_gold_examples_df

In [ ]:
# This code is selecting a random sample of 10 rows from the sample_reviews_gold_examples_df DataFrame
# ensuring that the same gold examples are selected for every session by setting the random_state parameter
# to a fixed value (in this case, 40).
# The selected rows are then converted to JSON format using the to_json() method with the orient='records' parameter.
gold_examples = (
        sample_reviews_gold_examples_df.loc[:, columns_to_select]
                                     .sample(10, random_state=40) #<- ensures that gold examples are the same for every session
                                     .to_json(orient='records')
)

In [ ]:
gold_examples

##**Question 5: Zero-shot Prompting for Sentiment Classification (8 Marks)**

(A) Write a system message for the model to classify the reviews into positive and negative (4 Marks)

(B) Calculate the Accuracy of the Model by comparing with the Actual Result (2 Marks)

(C) Evaluate on Gold Reviews (2 Marks)

In [ ]:
sample_reviews_examples_df

In [ ]:
sample_reviews_examples_df.review_sentiment.value_counts()

In [ ]:
def generate_llama_response( system_message, input_text , temp ):

    # Combine user_prompt and system_message to create the prompt
    prompt = f"[INST]{system_message}\n{input_text}[/INST]"

    print (prompt)

    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=4500,
        temperature=temp,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop="[/INST]",
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"]
    return response_text



###**(A) Write System Message Here**

In [ ]:
system_message_zero_shot = """
<<SYS>>You are a model that helps with classifying customer reviews as positive or negative. Including a list of customer reviews, classify them as either positive or negative<</SYS>>
"""

In [ ]:
user_message_template = "```{review}```"

In [ ]:
input_text = """
I am extremely disappointed with the service I received at your store!
The staff was rude and unhelpful, showing no regard for my concerns.
Not only did they ignore my requests for assistance, but they also had the audacity to speak to me condescendingly.
It's clear that your company values profit over customer satisfaction.
I will never shop here again and will make sure to spread the word about my awful experience.
You've lost a loyal customer, and I hope others steer clear of your establishment!
"""

In [ ]:
response = generate_llama_response( system_message_zero_shot , input_text , 0.1 )

In [ ]:
response

##*Measuring Prompt Performance*

In [ ]:
sample_reviews = sample_reviews_examples_df.review_text.values

In [ ]:
sentiment_predictions = []

In [ ]:
for sample_review in tqdm(sample_reviews):
    try:
        sentiment_predictions.append(generate_llama_response( system_message_zero_shot , sample_review , 0.1 ))
    except Exception as e:
        print(e)
        sentiment_predictions.append("")

  0%|          | 0/20 [00:00<?, ?it/s]Llama.generate: prefix-match hit


[INST]
<<SYS>>You are a model that helps with classifying customer reviews as positive or negative. Including a list of customer reviews, classify them as either positive or negative<</SYS>>

Awesome power bank, it charges my phone very fast and lasts for a long time. It is also very compact and lightweight, easy to carry around. It has a LED indicator that shows the battery level and a dual USB port that can charge two devices at the same time. It also has a low power mode that can charge small devices like earphones and smartwatches. I am very satisfied with this product.[/INST]



llama_print_timings:        load time =    1023.18 ms
llama_print_timings:      sample time =      26.17 ms /    45 runs   (    0.58 ms per token,  1719.46 tokens per second)
llama_print_timings: prompt eval time =     396.49 ms /    91 tokens (    4.36 ms per token,   229.51 tokens per second)
llama_print_timings:        eval time =    2360.36 ms /    44 runs   (   53.64 ms per token,    18.64 tokens per second)
llama_print_timings:       total time =    2810.96 ms /   135 tokens
  5%|▌         | 1/20 [00:02<00:53,  2.82s/it]Llama.generate: prefix-match hit


[INST]
<<SYS>>You are a model that helps with classifying customer reviews as positive or negative. Including a list of customer reviews, classify them as either positive or negative<</SYS>>

I bought this phone mainly for its much-hyped camera, but I was very disappointed with the results. The pictures are blurry, grainy, and overexposed. The zoom is also very bad, it makes the pictures look pixelated and distorted. The night mode is also useless, it makes the pictures look dark and noisy. The video quality is also very poor, it lags and stutters. Battery is also not very good. Only good thing is display. I expected much better from a flagship phone.[/INST]



llama_print_timings:        load time =    1023.18 ms
llama_print_timings:      sample time =     101.54 ms /   169 runs   (    0.60 ms per token,  1664.30 tokens per second)
llama_print_timings: prompt eval time =     371.82 ms /   118 tokens (    3.15 ms per token,   317.36 tokens per second)
llama_print_timings:        eval time =    9358.54 ms /   168 runs   (   55.71 ms per token,    17.95 tokens per second)
llama_print_timings:       total time =    9952.96 ms /   286 tokens
 10%|█         | 2/20 [00:12<02:06,  7.03s/it]Llama.generate: prefix-match hit


...


llama_print_timings:        load time =    1023.18 ms
llama_print_timings:      sample time =      71.43 ms /   112 runs   (    0.64 ms per token,  1567.97 tokens per second)
llama_print_timings: prompt eval time =     511.61 ms /   151 tokens (    3.39 ms per token,   295.15 tokens per second)
llama_print_timings:        eval time =    7139.50 ms /   111 runs   (   64.32 ms per token,    15.55 tokens per second)
llama_print_timings:       total time =    7805.30 ms /   262 tokens
 90%|█████████ | 18/20 [02:42<00:17,  8.62s/it]Llama.generate: prefix-match hit


[INST]
<<SYS>>You are a model that helps with classifying customer reviews as positive or negative. Including a list of customer reviews, classify them as either positive or negative<</SYS>>

Great for listening to music or podcasts. Good sound quality, battery life and charging speed. Quite comfortable too. Highly recommended![/INST]



llama_print_timings:        load time =    1023.18 ms
llama_print_timings:      sample time =      43.58 ms /    77 runs   (    0.57 ms per token,  1766.70 tokens per second)
llama_print_timings: prompt eval time =     168.54 ms /    34 tokens (    4.96 ms per token,   201.73 tokens per second)
llama_print_timings:        eval time =    4882.32 ms /    76 runs   (   64.24 ms per token,    15.57 tokens per second)
llama_print_timings:       total time =    5145.93 ms /   110 tokens
 95%|█████████▌| 19/20 [02:47<00:07,  7.58s/it]Llama.generate: prefix-match hit


[INST]
<<SYS>>You are a model that helps with classifying customer reviews as positive or negative. Including a list of customer reviews, classify them as either positive or negative<</SYS>>

I love this power bank - it's amazing. It can charge my phone several times and it is very compact and portable. It also has a fast charging feature, which is very convenient. The power bank is durable and has a sleek design. I am very satisfied with this product and I would highly recommend it.[/INST]



llama_print_timings:        load time =    1023.18 ms
llama_print_timings:      sample time =      77.47 ms /   123 runs   (    0.63 ms per token,  1587.69 tokens per second)
llama_print_timings: prompt eval time =     378.30 ms /    72 tokens (    5.25 ms per token,   190.33 tokens per second)
llama_print_timings:        eval time =    7917.25 ms /   122 runs   (   64.90 ms per token,    15.41 tokens per second)
llama_print_timings:       total time =    8452.29 ms /   194 tokens
100%|██████████| 20/20 [02:55<00:00,  8.79s/it]


In [ ]:
sample_reviews_examples_df["sentiment_prediction"] = sentiment_predictions

In [ ]:
sample_reviews_examples_df.sentiment_prediction.value_counts()

In [ ]:
sample_reviews_examples_df

In [ ]:
#This code extracts only Positive or Negative from the llama response for classification the customer reviews
sample_reviews_examples_df['sentiment_prediction_cleaned'] = sample_reviews_examples_df['sentiment_prediction'].str.extract(r'(Positive|Negative)', flags=re.IGNORECASE)

In [ ]:
# This is to apply lower case to all predicted sentiment values (e.g. Postive > positive)
sample_reviews_examples_df['sentiment_prediction_cleaned'] = sample_reviews_examples_df['sentiment_prediction_cleaned'].apply(lambda x: str(x).lower())

In [ ]:
# View the updated dataframe
sample_reviews_examples_df

 ### **(B) Calculate Accuracy Here**

In [ ]:
from sklearn.metrics import accuracy_score
#This will calculate the accuracy scope of the zero shot predictions
accuracy_zero_shot =  accuracy_score(sample_reviews_examples_df['review_sentiment'],
                                     sample_reviews_examples_df['sentiment_prediction_cleaned'])

In [ ]:
#view the zero shot accuracy
accuracy_zero_shot

In [ ]:
#Define the function to calculate
def evaluate_prompt(prompt, gold_examples, user_message_template):

    """
    Return the micro-F1 score for predictions on gold examples.
    For each example, we make a prediction using the prompt. Gold labels and
    model predictions are aggregated into lists and compared to compute the
    F1 score.

    Args:
        prompt (List): list of messages in the Open AI prompt format
        gold_examples (str): JSON string with list of gold examples
        user_message_template (str): string with a placeholder for movie reviews

    Output:
        micro_f1_score (float): Micro-F1 score computed by comparing model predictions
                                with ground truth
    """

    model_predictions, ground_truths = [], []

    for example in json.loads(gold_examples):
        gold_input = example['review_text']
        user_input = [
            {
                'role':'user',
                'content': user_message_template.format(review=gold_input)
            }
        ]

        try:
            prediction = generate_llama_response( prompt , user_input , 0.1 )
            print("prediction : " + prediction + "\n")

            prediction_extracted = re.search(r'(Positive|Negative)', prediction, flags=re.IGNORECASE).group().lower()
            # sentiment = match.group().lower() if match else "Sentiment not found."
            print("model_prediction : " + prediction_extracted + "\n")

            model_predictions.append(prediction_extracted) # <- removes extraneous white space and lowercases output
            ground_truths.append(example['review_sentiment'].lower())

            print("ground truth : " + example['review_sentiment'].lower() + "\n")

        except Exception as e:
            continue

    micro_f1_score = f1_score(ground_truths, model_predictions, average="micro")

    return micro_f1_score

### **(C) Evaluate the Prompt Here**

In [ ]:
#generate the f1 score
f1_score_zero_shot = evaluate_prompt(system_message_zero_shot, gold_examples, user_message_template)

##**Question 6: Few-shot Prompting for Sentiment Classification (8 Marks)**

(A) Write a system message for the model to classify the reviews into positive and negative. (4 Marks)

(B) Calculate the Accuracy of the Model by comparing with the Actual Result (2 Marks)

(C) Evaluate on Gold Reviews (2 Marks)

In [ ]:
sample_reviews_examples_df.review_sentiment.value_counts()

### **(A) Write a system message for the model to classify the reviews into positive and negative**


In [ ]:
system_message_few_shot = """
<<SYS>>You are a helpful agent that specializes in classifying customer reviews as either "Positive" or "Negative". Do not include any additional information or context in your response. Here are some examples:<</SYS>>
"""

In [ ]:
def create_examples_with_seed(dataset, n=2, random_seed=None):
    """
    Return two DataFrames with randomized examples of size 2n with two classes.
    Create subsets of each class, choose random samples from the subsets,
    merge and randomize the order of samples in the merged list.
    Each run of this function creates a different random sample of examples
    chosen from the training data.

    Args:
        dataset (DataFrame): A DataFrame with examples (text + label)
        n (int): number of examples of each class to be selected
        random_seed (int): seed for reproducibility (default is None)

    Output:
        few_shot_examples_df (DataFrame): A DataFrame with examples in random order
        new_df (DataFrame): A new DataFrame excluding selected examples
    """

    positive_reviews = (dataset.review_sentiment == 'Positive')
    negative_reviews = (dataset.review_sentiment == 'Negative')
    columns_to_select = ['review_text', 'review_sentiment']

    # Set a fixed random seed for reproducibility
    np.random.seed(random_seed)

    positive_examples = dataset.loc[positive_reviews, columns_to_select].sample(n)
    negative_examples = dataset.loc[negative_reviews, columns_to_select].sample(n)

    few_shot_examples_df = pd.concat([positive_examples, negative_examples])
    # sampling without replacement is equivalent to random shuffling
    few_shot_examples_df = few_shot_examples_df.sample(2 * n, replace=False)

    # Create a new DataFrame excluding selected examples
    new_df = dataset.drop(index=few_shot_examples_df.index)

    return few_shot_examples_df, new_df


In [ ]:
user_message_template = "```{review}```"

In [ ]:
def generate_llama_response( system_message ,  few_shot_examples  , new_review , temp ):

    # Combine user_prompt and system_message to create the prompt
    prompt = f"[INST]{system_message}\n{few_shot_examples}\n{'user'}: {user_message_template.format(review=new_review)}[/INST]"


    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=256,
        temperature=temp,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"]
    return response_text



**Measuring Prompt Performance**

In [ ]:
accuracy_list = []

### **(B) Calculate Accuracy and Mean Accuracy Here**

In this code snippet, match.group() returns the matched substring that was found by the regular expression pattern (positive|negative) in the prediction string.
For example, if prediction is "This movie was terrible and I hated it - negative", then match.group() will return the string "negative". If there is no match found, match will be None, and the code will append a default value (in this case, "unknown") to the sentiment_predictions_cleaned list.

In [ ]:
for i in range(5):
    few_shot_examples_df, sample_reviews_df = create_examples_with_seed(sample_reviews_examples_df, n=2, random_seed=i)


    few_shot_examples = few_shot_examples_df.to_json(orient='records')

    few_shot_examples_str = json.dumps(few_shot_examples)
    sample_reviews = sample_reviews_df.review_text.values

    sentiment_predictions = []

    for sample_review in tqdm(sample_reviews):
        try:
            response = generate_llama_response(system_message_few_shot, few_shot_examples_str, sample_review, 0.1)
            sentiment_predictions.append(response)
        except Exception as e:
            print(e)
            sentiment_predictions.append("")  # Default value for failed predictions

    sentiment_groundtruth = sample_reviews_df.review_sentiment.str.lower().values

    # Cleaning and validating predictions
    sentiment_predictions_cleaned = []
    for prediction in sentiment_predictions:
        if prediction:  # Check if prediction is not empty
            match = re.search(r'(positive|negative)', prediction, flags=re.IGNORECASE)
            if match:
                sentiment_predictions_cleaned.append(match.group().lower())
            else:
                sentiment_predictions_cleaned.append("unknown")  # Or any other default value for unmatched patterns
        else:
            sentiment_predictions_cleaned.append("unknown")  # Or any other default value for empty predictions

    # (B) Calculate Accuracy Here :
    num_correct = sum(pred == gt for pred, gt in zip(sentiment_predictions_cleaned, sentiment_groundtruth))
    total = len(sentiment_predictions_cleaned)
    res = num_correct / total

    # Calculated Accuracy will be appended to Accuracy List
    accuracy_list.append(res)

  0%|          | 0/16 [00:00<?, ?it/s]Llama.generate: prefix-match hit

llama_print_timings:        load time =    1023.18 ms
llama_print_timings:      sample time =       1.06 ms /     2 runs   (    0.53 ms per token,  1892.15 tokens per second)
llama_print_timings: prompt eval time =    1223.68 ms /   567 tokens (    2.16 ms per token,   463.35 tokens per second)
llama_print_timings:        eval time =      51.82 ms /     1 runs   (   51.82 ms per token,    19.30 tokens per second)
llama_print_timings:       total time =    1280.02 ms /   568 tokens
  6%|▋         | 1/16 [00:01<00:19,  1.30s/it]Llama.generate: prefix-match hit

llama_print_timings:        load time =    1023.18 ms
llama_print_timings:      sample time =     177.76 ms /   256 runs   (    0.69 ms per token,  1440.12 tokens per second)
llama_print_timings: prompt eval time =     425.16 ms /   120 tokens (    3.54 ms per token,   282.25 tokens per second)
llama_print_timings:        eval time =   18150.87 ms /   255 runs

In [ ]:
accuracy_list

[0.0, 0.0, 0.0, 0.0, 0.0]

In [ ]:
import statistics
#calculate the accuracy mean of the list of accuracies
mean_accuracy = statistics.mean(accuracy_list)


In [ ]:
mean_accuracy

0.0

### **(C) Evaluation Few Shot Prompting**

In [ ]:
def evaluate_prompt(prompt, gold_examples, user_message_template):

    """
    Return the micro-F1 score for predictions on gold examples.
    For each example, we make a prediction using the prompt. Gold labels and
    model predictions are aggregated into lists and compared to compute the
    F1 score.

    Args:
        prompt (List): list of messages in the Open AI prompt format
        gold_examples (str): JSON string with list of gold examples
        user_message_template (str): string with a placeholder for movie reviews

    Output:
        micro_f1_score (float): Micro-F1 score computed by comparing model predictions
                                with ground truth
    """

    model_predictions, ground_truths = [], []

    for example in json.loads(gold_examples):
        gold_input = example['review_text']
        user_input = [
            {
                'role':'user',
                'content': user_message_template.format(review=gold_input)
            }
        ]

        try:
            prediction = generate_llama_response( prompt ,few_shot_examples, user_input , 0.1 )
            print("prediction : " + prediction + "\n")

            prediction_extracted = re.search(r'(Positive|Negative)', prediction, flags=re.IGNORECASE).group().lower()
            # sentiment = match.group().lower() if match else "Sentiment not found."
            print("model_prediction : " + prediction_extracted + "\n")

            model_predictions.append(prediction_extracted) # <- removes extraneous white space and lowercases output
            ground_truths.append(example['review_sentiment'].lower())

            print("ground truth : " + example['review_sentiment'].lower() + "\n")

        except Exception as e:
            continue

    micro_f1_score = f1_score(ground_truths, model_predictions, average="micro")

    return micro_f1_score

### **(C) Evaluate Prompt Here**

In [ ]:
#Evaluate the Few Shot Prompt
few_shot_f1_score = evaluate_prompt(system_message_few_shot, gold_examples, user_message_template)
print (few_shot_f1_score)

#*Aspect-based Sentiment Classification*

##*Data Preprocessing*

##**Question 7: Data Preprocessing for Aspect Based Sentiment Analysis (5 Marks)**

(A) Read the CSV File (2 Marks)

(B) Divide the Dataset Based on product_type into 22 and Train examples and 8 Test examples (Gold Reviews), such that Test Data contains 2 examples of each of the 4 product_type (3 Marks)

In [ ]:
# The data is stored on my Google Colab
# Mouting Drive and Reading the CSV
drive.mount('/content/drive')
# Load a CSV File containing Dataset of 30 Reviews
sample_reviews_df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/GenAI - 2024/Final Project/Final_Project_customer_reviews_dataset.csv")

Mounted at /content/drive


In [ ]:
sample_reviews_df["aspects_review"][0]

'{ battery : Positive , keyboard : Positive, display : Positive }'

In [ ]:
laptop_reviews = sample_reviews_df[sample_reviews_df['product_type'] == 'Laptop']
headphones_reviews = sample_reviews_df[sample_reviews_df['product_type'] == 'Headphones']
smartphone_reviews = sample_reviews_df[sample_reviews_df['product_type'] == 'Smartphone']
power_bank_reviews = sample_reviews_df[sample_reviews_df['product_type'] == 'Power Bank']


laptop_gold_examples = laptop_reviews.sample(2, random_state=40)
headphones_gold_examples = headphones_reviews.sample(2, random_state=40)
smartphone_gold_examples = smartphone_reviews.sample(2, random_state=40)
power_bank_gold_examples = power_bank_reviews.sample(2, random_state=40)



# Concatenate positive and negative gold examples
sample_reviews_gold_examples_df = pd.concat([laptop_gold_examples, headphones_gold_examples,smartphone_gold_examples,power_bank_gold_examples])

# Create the training set by excluding gold examples
sample_reviews_examples_df = sample_reviews_df.drop(index=sample_reviews_gold_examples_df.index)

# Convert gold examples to JSON
columns_to_select = ['review_text', 'product_type','aspects_review']
gold_examples_json = sample_reviews_gold_examples_df[columns_to_select].to_json(orient='records')

# Print the first record from the JSON
print(json.loads(gold_examples_json)[0])

# Print the shapes of the datasets
print("Training Set Shape:", sample_reviews_examples_df.shape)
print("Gold Examples Shape:", sample_reviews_gold_examples_df.shape)


{'review_text': "Originally bought it for my work, quite happy with it so far! Fast, reliable, easy to use and has a good webcam. Display is good and battery backup is also great. The keyboard is a joy to type on, gives me the old typewriter vibes! Quickly become my main laptop for everyday use, and I'm very satisfied with my purchase.", 'product_type': 'Laptop', 'aspects_review': '{ battery : Positive , keyboard : Positive , display : Positive }'}
Training Set Shape: (22, 11)
Gold Examples Shape: (8, 11)


In [ ]:
sample_reviews_gold_examples_df

,customer_id,customer_name,product_name,product_type,review_date,rating,review_sentiment,review_text,aspects_review,response,summary
27,CID053,Pooja Jain,Dell Inspiron 15,Laptop,12/31/2023,4,Positive,"Originally bought it for my work, quite happy ...","{ battery : Positive , keyboard : Positive , d...",Thank you for sharing your positive experience...,The customer is satisfied with their work lapt...
14,CID004,Bradley Wiggins,Dell Inspiron 15,Laptop,05/10/2024,5,Positive,"This is a great laptop, I am very happy with i...","{ battery : Positive , keyboard : Positive , d...",Thank you for your positive feedback on our la...,"The customer is satisfied with their laptop, s..."
23,CID075,Jack Ryder,JBL Tune 500BT,Headphones,05/05/2023,5,Positive,Great for listening to music or podcasts. Good...,"{ sound : Positive , comfort : Positive , char...",Thank you for your positive review of our head...,Customer praises headphones for good sound qua...
13,CID060,Ketan Kopikar,JBL Tune 500BT,Headphones,01/06/2024,2,Negative,"I bought these headphones just 2 weeks ago, an...","{ sound : Negative , comfort : Negative , char...",I'm sorry to hear about your negative experien...,"The customer is unhappy with their headphones,..."
16,CID030,Rachel Smith,Samsung Galaxy S21,Smartphone,08/28/2023,4,Positive,"Good phone, I am satisfied with it. The phone ...","{ display : Positive , camera : Positive , Bat...",Thank you for your positive review of the phon...,"The customer is satisfied with their phone, pr..."
3,CID032,Jamal Johnson,Samsung Galaxy S21,Smartphone,03/03/2023,2,Negative,I bought this phone mainly for its much-hyped ...,"{ display : Positive , camera : Negative , Bat...",I'm truly sorry to hear that the camera perfor...,The user expressed disappointment with the pho...
24,CID046,Harvey Stark,Mi Power Bank 3i,Power Bank,07/19/2023,5,Positive,I love this power bank - it's amazing. It can ...,"{ battery : Positive , charging : Positive }",Thank you for your enthusiastic feedback on ou...,The customer praises the power bank's capacity...
5,CID012,Rafael Fernandez,Mi Power Bank 3i,Power Bank,01/13/2024,1,Negative,"This is a very bad power bank, it does not cha...","{ battery : Negative , charging : Negative }",I'm deeply concerned to hear about your experi...,"The customer reviews a poor power bank, statin..."


In [ ]:
gold_examples = (
        sample_reviews_gold_examples_df.loc[:, columns_to_select]
                                     .sample(8, random_state=40) #<- ensures that gold examples are the same for every session
                                     .to_json(orient='records')
)

In [ ]:
user_message_template = """{review}"""

In [ ]:
sample_reviews_examples_df[sample_reviews_examples_df['product_type']=="Smartphone"].aspects_review.value_counts()

aspects_review
{ display : Negative , camera : Negative , Battery : Negative }    1
{ display : Negative , camera : Positive , Battery : Negative }    1
{ display : Negative , camera : Positive , Battery : Positive }    1
Name: count, dtype: int64

## **Question 8: Zero-shot Prompting for ABSA (8 Marks)**

(A) Write System Prompt to get the Aspect Based Sentiment for each of the review in format mentioned in csv. You will need to first identify the product , then based on the product , you will know the aspects for which you need sentiments (4 Marks)

(B) Find Accuracy by comparing to actual predictions (2 Marks)

(C) Evaluate the model (2 Marks)

###**(A) Write your Prompt Here**

In [ ]:
zero_shot_system_message = """
<<SYS>>
You are a sophisticated AI trained to analyze customer reviews for four product categories:
1. Headphone
2. Laptop
3. Smartphone
4. Power Bank.

These are the aspects for each product category:
For Laptop: battery, keyboard, display
For Headphones: sounds, comfort, charging
For Power Bank: power, charging
For Smartphone: camera, display, battery

For each of the four product categories, determine the sentiment (positive or negative) for each of their aspects.
Finally, format the analysis results by listing each aspect and its sentiment separated by a semicolon and all aspects separated by commas, like the following:

battery : positive, keyboard : negative, display: neutral

Ensure to handle each review independently and provide the sentiment analysis based on the content of the following review: <</SYS>>
"""

In [ ]:
def generate_llama_response( system_message, input_text , temp ):

    # Combine user_prompt and system_message to create the prompt
    prompt = f"[INST]{system_message}\n```{input_text}```[/INST]"

    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=2500,
        temperature=temp,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"]
    return response_text



In [ ]:
input_text = """
I bought this laptop for my son who is studying engineering. He is very happy with it. It has a good battery life, fast performance, and a sleek design.
The keyboard is comfortable and the screen is bright. The laptop came with a one-year warranty and a free antivirus software. I think it is a great value for money.
"""

In [ ]:
response = generate_llama_response( zero_shot_system_message , input_text , 2.4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =      15.98 ms /    21 runs   (    0.76 ms per token,  1313.98 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =    1320.43 ms /    21 runs   (   62.88 ms per token,    15.90 tokens per second)
llama_print_timings:       total time =    1352.95 ms /    21 tokens


In [ ]:
response

'\n\n<pre class="output">battery : positive, keyboard: negative</pre>'

###*Measuring Prompt Performance*

In [ ]:
sample_reviews = sample_reviews_examples_df.review_text.values

In [ ]:
sentiment_predictions = []

In [ ]:
generate_llama_response( zero_shot_system_message , input_text , 1.5 )

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =       0.69 ms /     1 runs   (    0.69 ms per token,  1445.09 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =      75.11 ms /     1 runs   (   75.11 ms per token,    13.31 tokens per second)
llama_print_timings:       total time =      77.63 ms /     1 tokens


''

In [ ]:
for sample_review in tqdm(sample_reviews):
    try:
        sentiment_predictions.append(generate_llama_response( zero_shot_system_message , sample_review , 1.5 ))
    except Exception as e:
        print(e)
        sentiment_predictions.append("")

  0%|          | 0/22 [00:00<?, ?it/s]Llama.generate: prefix-match hit

llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =     184.11 ms /   292 runs   (    0.63 ms per token,  1586.04 tokens per second)
llama_print_timings: prompt eval time =     414.93 ms /    79 tokens (    5.25 ms per token,   190.39 tokens per second)
llama_print_timings:        eval time =   20389.07 ms /   291 runs   (   70.07 ms per token,    14.27 tokens per second)
llama_print_timings:       total time =   21225.67 ms /   370 tokens
  5%|▍         | 1/22 [00:21<07:26, 21.24s/it]Llama.generate: prefix-match hit

llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =       0.54 ms /     1 runs   (    0.54 ms per token,  1865.67 tokens per second)
llama_print_timings: prompt eval time =     444.62 ms /   102 tokens (    4.36 ms per token,   229.41 tokens per second)
llama_print_timings:        eval time =       0.00 ms /     1 runs

In [ ]:
sentiment_predictions

["\n\n[OUTS] \nFor each product category (Laptop, Headphone, Smartphone and Power Bank), we will compute sentiment by counting number of reviews in which that aspect has been mentioned. We need to make sure these numbers are computed separately because one review can talk about multiple aspects! And we want to know what the sentiments are for each respective product category based on those numbers \nSo we have the following data:\n```Python \n(laptop_battery, laptop_keyboard , laptoop_display) = ('positive', 'negative', 'neutral')\n\n(headphones_sound, headphones_comfort, headphone_charging) = ('positive', 'positive', 'neural')\n \n(smartphone_camera, smartphone_display , smarthpobe_battery) = (10, 2, -3) # these are numbers that indicate positive or negative sentiments.\n```\nSince there is no powerbank category in this problem description, we will set the sentiments for each of its aspects to 'neutral'.\nIn addition we have a very simple logic when computing sentiments based on aspec

In [ ]:
sample_reviews_examples_df["sentiment_prediction"] = sentiment_predictions

In [ ]:
pattern = r'\[\s*.*?\s*:\s*\{.*?\}\s*\]'

In [ ]:
# Function to extract the sentiment data
def extract_sentiment(text):
    text=str(text)
    match = re.findall(pattern, text)
    return match[0] if match else None

# Apply the function to each row in the DataFrame
sample_reviews_examples_df['extracted_sentiment'] = sample_reviews_examples_df['sentiment_prediction'].apply(extract_sentiment)


In [ ]:
sample_reviews_examples_df

,customer_id,customer_name,product_name,product_type,review_date,rating,review_sentiment,review_text,aspects_review,response,summary,sentiment_prediction,extracted_sentiment
0,CID041,Aisha Patel,Dell Inspiron 15,Laptop,01/01/2024,5,Positive,I bought this laptop for my son who is studyin...,"{ battery : Positive , keyboard : Positive, di...",It's fantastic to hear that the laptop you pur...,"The user purchased a laptop for their son, who...",\n\n[OUTS] \nFor each product category (Laptop...,None
1,CID011,Liam Thomson,JBL Tune 500BT,Headphones,01/05/2024,1,Negative,I was very disappointed with these headphones....,"{ sound : Negative , comfort : Negative , char...",I'm truly sorry to hear about your disappointi...,The user expressed disappointment with poor so...,,None
2,CID034,Maria García,Mi Power Bank 3i,Power Bank,01/10/2023,4,Positive,"Awesome power bank, it charges my phone very f...","{ battery : Positive , charging : Positive }",Thank you for your positive review of our powe...,The user praises the power bank for its fast c...,,None
4,CID051,Emily Nguyen,JBL Tune 500BT,Headphones,05/25/2023,5,Positive,"I love these headphones, they are amazing. The...","{ sound : Positive , comfort : Positive , char...",Thank you for your wonderful feedback on our h...,The user praises the headphones for their exce...,\n\n**Explanation:** This problem uses an Inst...,None
6,CID043,Sofia Chen,Dell Inspiron 15,Laptop,05/20/2023,1,Negative,"I bought this laptop a month ago, and it's alr...","{ battery : Negative , keyboard : Negative , d...",I'm sorry to hear about your laptop issues and...,"The customer reports issues with their laptop,...",,None
7,CID063,Michael Lee,Dell Inspiron 15,Laptop,02/09/2023,2,Negative,This is a below average laptop. It has inconsi...,"{ battery : Negative , keyboard : Negative , d...",I'm sorry to hear that your laptop isn't meeti...,The customer reviews a below-average laptop wi...,\n\n**Input File:** [laptop-sentiment.txt](htt...,None
8,CID029,Olivia Rodriguez,JBL Tune 500BT,Headphones,01/02/2024,1,Negative,"I bought these headphones a week ago, and they...","{ sound : Negative , comfort : Negative , char...",I apologize for the issues you're experiencing...,The customer purchased poor-quality headphones...,\n,None
9,CID080,Karan Singh,JBL Tune 500BT,Headphones,02/22/2023,5,Positive,These are the best headphones I have ever used...,"{ sound : Positive , comfort : Positive , char...",Thank you for sharing your positive experience...,The customer praises the headphones for their ...,,None
10,CID007,David Kim,Dell Inspiron 15,Laptop,05/03/2023,1,Negative,"I hate this laptop, it is a nightmare. I bough...","{ battery : Negative , keyboard : Negative , d...",I'm truly sorry to hear about the difficulties...,The customer reviews a laptop with poor softwa...,\n,None
11,CID040,Reena Gupta,Mi Power Bank 3i,Power Bank,10/02/2023,5,Positive,Recently tried a power bank sharing service - ...,"{ battery : Positive , charging : Negative }",Thank you for your feedback on the power bank ...,The customer reviews a power bank sharing serv...,,None


In [ ]:
# Function to extract product type
def extract_product_type(text):
    rext=str(text)
    match = re.search(r'\[([^:]+)', rext)
    return match.group(1).strip() if match else None

# Function to extract aspect-based sentiment
def extract_aspect_based_sentiment(text):
    text=str(text)
    match = re.search(r'\{([^}]+)\}', text)
    return "{ " + match.group(1).strip() + " }" if match else None

# Apply the functions to each row in the DataFrame
sample_reviews_examples_df['predicted_product_type'] = sample_reviews_examples_df['extracted_sentiment'].apply(extract_product_type)
sample_reviews_examples_df['predicted_aspect_based_sentiment'] = sample_reviews_examples_df['extracted_sentiment'].apply(extract_aspect_based_sentiment)


In [ ]:
sample_reviews_examples_df

,customer_id,customer_name,product_name,product_type,review_date,rating,review_sentiment,review_text,aspects_review,response,summary,sentiment_prediction,extracted_sentiment,predicted_product_type,predicted_aspect_based_sentiment
0,CID041,Aisha Patel,Dell Inspiron 15,Laptop,01/01/2024,5,Positive,I bought this laptop for my son who is studyin...,"{ battery : Positive , keyboard : Positive, di...",It's fantastic to hear that the laptop you pur...,"The user purchased a laptop for their son, who...",\n\n[OUTS] \nFor each product category (Laptop...,None,None,None
1,CID011,Liam Thomson,JBL Tune 500BT,Headphones,01/05/2024,1,Negative,I was very disappointed with these headphones....,"{ sound : Negative , comfort : Negative , char...",I'm truly sorry to hear about your disappointi...,The user expressed disappointment with poor so...,,None,None,None
2,CID034,Maria García,Mi Power Bank 3i,Power Bank,01/10/2023,4,Positive,"Awesome power bank, it charges my phone very f...","{ battery : Positive , charging : Positive }",Thank you for your positive review of our powe...,The user praises the power bank for its fast c...,,None,None,None
4,CID051,Emily Nguyen,JBL Tune 500BT,Headphones,05/25/2023,5,Positive,"I love these headphones, they are amazing. The...","{ sound : Positive , comfort : Positive , char...",Thank you for your wonderful feedback on our h...,The user praises the headphones for their exce...,\n\n**Explanation:** This problem uses an Inst...,None,None,None
6,CID043,Sofia Chen,Dell Inspiron 15,Laptop,05/20/2023,1,Negative,"I bought this laptop a month ago, and it's alr...","{ battery : Negative , keyboard : Negative , d...",I'm sorry to hear about your laptop issues and...,"The customer reports issues with their laptop,...",,None,None,None
7,CID063,Michael Lee,Dell Inspiron 15,Laptop,02/09/2023,2,Negative,This is a below average laptop. It has inconsi...,"{ battery : Negative , keyboard : Negative , d...",I'm sorry to hear that your laptop isn't meeti...,The customer reviews a below-average laptop wi...,\n\n**Input File:** [laptop-sentiment.txt](htt...,None,None,None
8,CID029,Olivia Rodriguez,JBL Tune 500BT,Headphones,01/02/2024,1,Negative,"I bought these headphones a week ago, and they...","{ sound : Negative , comfort : Negative , char...",I apologize for the issues you're experiencing...,The customer purchased poor-quality headphones...,\n,None,None,None
9,CID080,Karan Singh,JBL Tune 500BT,Headphones,02/22/2023,5,Positive,These are the best headphones I have ever used...,"{ sound : Positive , comfort : Positive , char...",Thank you for sharing your positive experience...,The customer praises the headphones for their ...,,None,None,None
10,CID007,David Kim,Dell Inspiron 15,Laptop,05/03/2023,1,Negative,"I hate this laptop, it is a nightmare. I bough...","{ battery : Negative , keyboard : Negative , d...",I'm truly sorry to hear about the difficulties...,The customer reviews a laptop with poor softwa...,\n,None,None,None
11,CID040,Reena Gupta,Mi Power Bank 3i,Power Bank,10/02/2023,5,Positive,Recently tried a power bank sharing service - ...,"{ battery : Positive , charging : Negative }",Thank you for your feedback on the power bank ...,The customer reviews a power bank sharing serv...,,None,None,None


In [ ]:
sample_reviews_examples_df

,customer_id,customer_name,product_name,product_type,review_date,rating,review_sentiment,review_text,aspects_review,response,summary,sentiment_prediction,extracted_sentiment,predicted_product_type,predicted_aspect_based_sentiment
0,CID041,Aisha Patel,Dell Inspiron 15,Laptop,01/01/2024,5,Positive,I bought this laptop for my son who is studyin...,"{ battery : Positive , keyboard : Positive, di...",It's fantastic to hear that the laptop you pur...,"The user purchased a laptop for their son, who...",\n\n[OUTS] \nFor each product category (Laptop...,None,None,None
1,CID011,Liam Thomson,JBL Tune 500BT,Headphones,01/05/2024,1,Negative,I was very disappointed with these headphones....,"{ sound : Negative , comfort : Negative , char...",I'm truly sorry to hear about your disappointi...,The user expressed disappointment with poor so...,,None,None,None
2,CID034,Maria García,Mi Power Bank 3i,Power Bank,01/10/2023,4,Positive,"Awesome power bank, it charges my phone very f...","{ battery : Positive , charging : Positive }",Thank you for your positive review of our powe...,The user praises the power bank for its fast c...,,None,None,None
4,CID051,Emily Nguyen,JBL Tune 500BT,Headphones,05/25/2023,5,Positive,"I love these headphones, they are amazing. The...","{ sound : Positive , comfort : Positive , char...",Thank you for your wonderful feedback on our h...,The user praises the headphones for their exce...,\n\n**Explanation:** This problem uses an Inst...,None,None,None
6,CID043,Sofia Chen,Dell Inspiron 15,Laptop,05/20/2023,1,Negative,"I bought this laptop a month ago, and it's alr...","{ battery : Negative , keyboard : Negative , d...",I'm sorry to hear about your laptop issues and...,"The customer reports issues with their laptop,...",,None,None,None
7,CID063,Michael Lee,Dell Inspiron 15,Laptop,02/09/2023,2,Negative,This is a below average laptop. It has inconsi...,"{ battery : Negative , keyboard : Negative , d...",I'm sorry to hear that your laptop isn't meeti...,The customer reviews a below-average laptop wi...,\n\n**Input File:** [laptop-sentiment.txt](htt...,None,None,None
8,CID029,Olivia Rodriguez,JBL Tune 500BT,Headphones,01/02/2024,1,Negative,"I bought these headphones a week ago, and they...","{ sound : Negative , comfort : Negative , char...",I apologize for the issues you're experiencing...,The customer purchased poor-quality headphones...,\n,None,None,None
9,CID080,Karan Singh,JBL Tune 500BT,Headphones,02/22/2023,5,Positive,These are the best headphones I have ever used...,"{ sound : Positive , comfort : Positive , char...",Thank you for sharing your positive experience...,The customer praises the headphones for their ...,,None,None,None
10,CID007,David Kim,Dell Inspiron 15,Laptop,05/03/2023,1,Negative,"I hate this laptop, it is a nightmare. I bough...","{ battery : Negative , keyboard : Negative , d...",I'm truly sorry to hear about the difficulties...,The customer reviews a laptop with poor softwa...,\n,None,None,None
11,CID040,Reena Gupta,Mi Power Bank 3i,Power Bank,10/02/2023,5,Positive,Recently tried a power bank sharing service - ...,"{ battery : Positive , charging : Negative }",Thank you for your feedback on the power bank ...,The customer reviews a power bank sharing serv...,,None,None,None


In [ ]:
sample_reviews_examples_df

,customer_id,customer_name,product_name,product_type,review_date,rating,review_sentiment,review_text,aspects_review,response,summary,sentiment_prediction,extracted_sentiment,predicted_product_type,predicted_aspect_based_sentiment
0,CID041,Aisha Patel,Dell Inspiron 15,Laptop,01/01/2024,5,Positive,I bought this laptop for my son who is studyin...,"{ battery : Positive , keyboard : Positive, di...",It's fantastic to hear that the laptop you pur...,"The user purchased a laptop for their son, who...",\n\n[OUTS] \nFor each product category (Laptop...,None,None,None
1,CID011,Liam Thomson,JBL Tune 500BT,Headphones,01/05/2024,1,Negative,I was very disappointed with these headphones....,"{ sound : Negative , comfort : Negative , char...",I'm truly sorry to hear about your disappointi...,The user expressed disappointment with poor so...,,None,None,None
2,CID034,Maria García,Mi Power Bank 3i,Power Bank,01/10/2023,4,Positive,"Awesome power bank, it charges my phone very f...","{ battery : Positive , charging : Positive }",Thank you for your positive review of our powe...,The user praises the power bank for its fast c...,,None,None,None
4,CID051,Emily Nguyen,JBL Tune 500BT,Headphones,05/25/2023,5,Positive,"I love these headphones, they are amazing. The...","{ sound : Positive , comfort : Positive , char...",Thank you for your wonderful feedback on our h...,The user praises the headphones for their exce...,\n\n**Explanation:** This problem uses an Inst...,None,None,None
6,CID043,Sofia Chen,Dell Inspiron 15,Laptop,05/20/2023,1,Negative,"I bought this laptop a month ago, and it's alr...","{ battery : Negative , keyboard : Negative , d...",I'm sorry to hear about your laptop issues and...,"The customer reports issues with their laptop,...",,None,None,None
7,CID063,Michael Lee,Dell Inspiron 15,Laptop,02/09/2023,2,Negative,This is a below average laptop. It has inconsi...,"{ battery : Negative , keyboard : Negative , d...",I'm sorry to hear that your laptop isn't meeti...,The customer reviews a below-average laptop wi...,\n\n**Input File:** [laptop-sentiment.txt](htt...,None,None,None
8,CID029,Olivia Rodriguez,JBL Tune 500BT,Headphones,01/02/2024,1,Negative,"I bought these headphones a week ago, and they...","{ sound : Negative , comfort : Negative , char...",I apologize for the issues you're experiencing...,The customer purchased poor-quality headphones...,\n,None,None,None
9,CID080,Karan Singh,JBL Tune 500BT,Headphones,02/22/2023,5,Positive,These are the best headphones I have ever used...,"{ sound : Positive , comfort : Positive , char...",Thank you for sharing your positive experience...,The customer praises the headphones for their ...,,None,None,None
10,CID007,David Kim,Dell Inspiron 15,Laptop,05/03/2023,1,Negative,"I hate this laptop, it is a nightmare. I bough...","{ battery : Negative , keyboard : Negative , d...",I'm truly sorry to hear about the difficulties...,The customer reviews a laptop with poor softwa...,\n,None,None,None
11,CID040,Reena Gupta,Mi Power Bank 3i,Power Bank,10/02/2023,5,Positive,Recently tried a power bank sharing service - ...,"{ battery : Positive , charging : Negative }",Thank you for your feedback on the power bank ...,The customer reviews a power bank sharing serv...,,None,None,None


In [ ]:
# sample_reviews_examples_df.to_csv('abc.csv')

In [ ]:
print(sample_reviews_examples_df['sentiment_prediction'])

0     \n\n[OUTS] \nFor each product category (Laptop...
1                                                      
2                                                      
4     \n\n**Explanation:** This problem uses an Inst...
6                                                      
7     \n\n**Input File:** [laptop-sentiment.txt](htt...
8                                                    \n
9                                                      
10                                                   \n
11                                                     
12                                                   \n
15                                                   \n
17                                                   \n
18                                                   \n
19                                                   \n
20                                                   \n
21                                                 \n\n
22                                              

In [ ]:
sample_reviews_examples_df.predicted_aspect_based_sentiment.value_counts()

Series([], Name: count, dtype: int64)

In [ ]:
def compute_combined_accuracy_ignoring_spaces_and_case(df):
    correct_predictions_count = 0

    # Function to normalize text by removing spaces and converting to lowercase
    def normalize_text(text):
        print(text)
        return ''.join(text.split()).lower() if text is not None else None

    # Iterate over each row to compare values
    for index, row in df.iterrows():
        # Normalize both actual and predicted values
        predicted_product_type = normalize_text(row['predicted_product_type'])
        actual_product_type = normalize_text(row['product_type'])
        predicted_aspect = normalize_text(row['predicted_aspect_based_sentiment'])
        actual_aspect = normalize_text(row['aspects_review'])

        if predicted_product_type == actual_product_type and \
           predicted_aspect == actual_aspect:
            correct_predictions_count += 1

    # Calculate combined accuracy
    combined_accuracy = (correct_predictions_count / len(df)) * 100

    return combined_accuracy



### **(B) Calculate Accuracy**

In [ ]:
# Sample usage with your DataFrame
combined_accuracy = compute_combined_accuracy_ignoring_spaces_and_case(sample_reviews_examples_df)

print(f"Combined Accuracy (Ignoring Spaces and Case): {combined_accuracy}%")


None
Laptop
None
{ battery : Positive , keyboard : Positive, display : Positive }
None
Headphones
None
{ sound : Negative , comfort : Negative , charging : Negative }
None
Power Bank
None
{ battery : Positive , charging : Positive }
None
Headphones
None
{ sound : Positive , comfort : Positive , charging : Positive }
None
Laptop
None
{ battery : Negative , keyboard : Negative , display : Negative }
None
Laptop
None
{ battery : Negative , keyboard : Negative , display : Positive }
None
Headphones
None
{ sound : Negative , comfort : Negative , charging : Negative }
None
Headphones
None
{ sound : Positive , comfort : Positive , charging : Negative }
None
Laptop
None
{ battery : Negative , keyboard : Negative , display : Positive }
None
Power Bank
None
{ battery : Positive , charging : Negative }
None
Smartphone
None
{ display : Negative , camera : Negative , Battery : Negative }
None
Power Bank
None
{ battery : Positive , charging : Positive }
None
Headphones
None
{ sound : Positive , comf

In [ ]:
def evaluate_prompt(prompt, gold_examples, user_message_template):

    """
    Return the micro-F1 score for predictions on gold examples.
    For each example, we make a prediction using the prompt. Gold labels and
    model predictions are aggregated into lists and compared to compute the
    F1 score.

    Args:
        prompt (List): list of messages in the Open AI prompt format
        gold_examples (str): JSON string with list of gold examples
        user_message_template (str): string with a placeholder for movie reviews

    Output:
        micro_f1_score (float): Micro-F1 score computed by comparing model predictions
                                with ground truth
    """

    model_predictions, ground_truths = [], []

    for example in json.loads(gold_examples):
        gold_input = example['review_text']
        user_input = [
            {
               user_message_template.format(review=gold_input)
            }
        ]

        try:
            prediction = generate_llama_response( prompt , user_input , 0.1 )


            # print("response" + response + "\n")
            # prediction = response["choices"][0]["text"]
            print("prediction : " + prediction + "\n")

            prediction_extracted = extract_sentiment(prediction)
            predicted_product_type = extract_product_type(prediction_extracted)
            predicted_aspect_based_sentiment = extract_aspect_based_sentiment(prediction_extracted)

            final_prediction = predicted_product_type + ":" + predicted_aspect_based_sentiment

            # sentiment = match.group().lower() if match else "Sentiment not found."
            print("model_prediction : " + final_prediction.lower() + "\n")

            model_predictions.append(final_prediction.lower()) # <- removes extraneous white space and lowercases output
            ground_truths.append( example['product_type'].lower() + ":" + example['aspects_review'].lower() )

            print("ground truth : " + example['product_type'].lower() + ":" + example['aspects_review'].lower() + "\n")

        except Exception as e:
            continue

    micro_f1_score = f1_score(ground_truths, model_predictions, average="micro")

    return micro_f1_score

### **(C) Evaluate Prompt**

In [ ]:
prompt_evaluation = evaluate_prompt(zero_shot_system_message, gold_examples, user_message_template)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =       1.20 ms /     2 runs   (    0.60 ms per token,  1669.45 tokens per second)
llama_print_timings: prompt eval time =     429.36 ms /   110 tokens (    3.90 ms per token,   256.20 tokens per second)
llama_print_timings:        eval time =      56.23 ms /     1 runs   (   56.23 ms per token,    17.78 tokens per second)
llama_print_timings:       total time =     490.14 ms /   111 tokens
Llama.generate: prefix-match hit


prediction : 





llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =       1.83 ms /     2 runs   (    0.91 ms per token,  1095.29 tokens per second)
llama_print_timings: prompt eval time =     404.12 ms /   123 tokens (    3.29 ms per token,   304.37 tokens per second)
llama_print_timings:        eval time =      65.50 ms /     1 runs   (   65.50 ms per token,    15.27 tokens per second)
llama_print_timings:       total time =     475.15 ms /   124 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =       0.68 ms /     1 runs   (    0.68 ms per token,  1472.75 tokens per second)
llama_print_timings: prompt eval time =     174.06 ms /    38 tokens (    4.58 ms per token,   218.31 tokens per second)
llama_print_timings:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_print_timings:       total time =     175.76 ms /    39 

prediction : 


prediction : 




llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =       1.02 ms /     2 runs   (    0.51 ms per token,  1970.44 tokens per second)
llama_print_timings: prompt eval time =     482.12 ms /   132 tokens (    3.65 ms per token,   273.79 tokens per second)
llama_print_timings:        eval time =      57.82 ms /     1 runs   (   57.82 ms per token,    17.29 tokens per second)
llama_print_timings:       total time =     542.28 ms /   133 tokens
Llama.generate: prefix-match hit


...


llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =       1.06 ms /     2 runs   (    0.53 ms per token,  1892.15 tokens per second)
llama_print_timings: prompt eval time =     429.43 ms /   103 tokens (    4.17 ms per token,   239.86 tokens per second)
llama_print_timings:        eval time =      71.85 ms /     1 runs   (   71.85 ms per token,    13.92 tokens per second)
llama_print_timings:       total time =     504.09 ms /   104 tokens
Llama.generate: prefix-match hit


prediction : 





llama_print_timings:        load time =    1050.09 ms
llama_print_timings:      sample time =       1.07 ms /     2 runs   (    0.53 ms per token,  1874.41 tokens per second)
llama_print_timings: prompt eval time =     383.69 ms /    76 tokens (    5.05 ms per token,   198.08 tokens per second)
llama_print_timings:        eval time =      57.80 ms /     1 runs   (   57.80 ms per token,    17.30 tokens per second)
llama_print_timings:       total time =     444.16 ms /    77 tokens


prediction : 




/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))


In [ ]:
prompt_evaluation

0.0

##**Question 9: Few-shot Prompting for ABSA (8 Marks)**

(A) Write System Prompt to get the Aspect Based Sentiment for each of the reviews in the format mentioned in the CSV. You will need to first identify the product, then based on the product, you will know the aspects for which you need sentiments. (4 Marks)

(B) Find Accuracy by comparing to actual predictions. (2 Marks)

(C) Evaluate the model (2 Marks)

###**(A) Write Prompt Here**

In [ ]:
few_shot_system_message = """
<<SYS>>__________<</SYS>>
"""

In [ ]:
def generate_llama_response( system_message ,  few_shot_examples  , new_review , temp ):

    # Combine user_prompt and system_message to create the prompt
    prompt = f"[INST]{system_message}\n{few_shot_examples}\n{'user'}: ```{user_message_template.format(review=new_review)}```[/INST]"


    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=256,
        temperature=temp,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"]
    return response_text



In [ ]:
def create_examples_with_seed(dataset, n=2, random_seed=None):
    """
    Return two DataFrames with randomized examples of size 2n with two classes.
    Create subsets of each class, choose random samples from the subsets,
    merge and randomize the order of samples in the merged list.
    Each run of this function creates a different random sample of examples
    chosen from the training data.

    Args:
        dataset (DataFrame): A DataFrame with examples (text + label)
        n (int): number of examples of each class to be selected
        random_seed (int): seed for reproducibility (default is None)

    Output:
        few_shot_examples_df (DataFrame): A DataFrame with examples in random order
        new_df (DataFrame): A new DataFrame excluding selected examples
    """

    laptop_reviews = (dataset.product_type == 'Laptop')
    headphone_reviews = (dataset.product_type == 'Headphones')
    power_bank_reviews = (dataset.product_type == 'Power Bank')
    smartphone_reviews = (dataset.product_type == 'Smartphone')
    columns_to_select = ['review_text', 'product_type','aspects_review']

    # Set a fixed random seed for reproducibility
    np.random.seed(random_seed)

    laptop_examples = dataset.loc[laptop_reviews, columns_to_select].sample(n)
    headphone_examples = dataset.loc[headphone_reviews, columns_to_select].sample(n)
    power_bank_examples = dataset.loc[power_bank_reviews, columns_to_select].sample(n)
    smartphone_examples = dataset.loc[smartphone_reviews, columns_to_select].sample(n)

    few_shot_examples_df = pd.concat([laptop_examples, headphone_examples, power_bank_examples, smartphone_examples])
    # sampling without replacement is equivalent to random shuffling
    few_shot_examples_df = few_shot_examples_df.sample( 4*n, replace=False)

    # Create a new DataFrame excluding selected examples
    new_df = dataset.drop(index=few_shot_examples_df.index)

    return few_shot_examples_df, new_df


In [ ]:
def compute_combined_aspect_and_product_accuracy(df):
    correct_predictions_count = 0

    # Function to parse aspect-based sentiment string into a dictionary
    def parse_aspects(aspect_string):
        aspect_string= str(aspect_string)
        aspect_string = re.sub(r'[{}]', '', aspect_string)  # Remove curly braces
        aspects = re.split(r',\s*', aspect_string)  # Split into individual aspects

        try:
         dict_res = dict(re.split(r'\s*:\s*', aspect) for aspect in aspects if aspect)
        except:
          dict_res={}

        return dict_res


    # Function to normalize text by removing spaces and converting to lowercase
    def normalize_text(text):
        return ''.join(text.split()).lower() if text is not None else None

    # Iterate over each row to compare product type and aspect-based sentiments
    for index, row in df.iterrows():
        predicted_product_type = normalize_text(row['predicted_product_type'])
        actual_product_type = normalize_text(row['product_type'])
        predicted_aspects = parse_aspects(row['predicted_aspect_based_sentiment'])
        actual_aspects = parse_aspects(row['aspects_review'])

        # Check if product type matches and all aspects match in sentiment
        if predicted_product_type == actual_product_type and \
          predicted_aspects != '' and \
          all(predicted_aspects.get(key, '').lower() == value.lower() for key, value in actual_aspects.items()):
            correct_predictions_count += 1

    # Calculate accuracy
    accuracy = (correct_predictions_count / len(df)) * 100
    return accuracy

###*Measuring Prompt Performance*

In [ ]:
user_message_template = "{review}"

In [ ]:
def extract_product_type(text):
    text=str(text)
    match = re.search(r'\[([^:]+)', text)
    return match.group(1).strip() if match else None

def extract_aspect_based_sentiment(text):
    text=str(text)
    match = re.search(r'\{([^}]+)\}', text)
    return "{ " + match.group(1).strip() + " }" if match else None

In [ ]:
accuracy_list = []

In [ ]:
sample_reviews_examples_df

###**(B) Calculate Accuracy and Mean Accuracy**

In [ ]:
for i in range(5):
    few_shot_examples_df ,sample_reviews_df = create_examples_with_seed(sample_reviews_examples_df, n=1 , random_seed = i)

    review_example1 = few_shot_examples_df.iloc[0].review_text
    review_example2 = few_shot_examples_df.iloc[1].review_text
    review_example3 = few_shot_examples_df.iloc[2].review_text
    review_example4 = few_shot_examples_df.iloc[3].review_text

    assistant_output_example1 = "[ " + few_shot_examples_df.iloc[0].product_type.lower() + " : " + few_shot_examples_df.iloc[0].aspects_review.lower() + " ]"
    assistant_output_example2 = "[ " + few_shot_examples_df.iloc[1].product_type.lower() + " : " + few_shot_examples_df.iloc[1].aspects_review.lower() + " ]"
    assistant_output_example3 = "[ " + few_shot_examples_df.iloc[2].product_type.lower() + " : " + few_shot_examples_df.iloc[2].aspects_review.lower() + " ]"
    assistant_output_example4 = "[ " + few_shot_examples_df.iloc[3].product_type.lower() + " : " + few_shot_examples_df.iloc[3].aspects_review.lower() + " ]"

    few_shot_examples = [
        {'role':'user', 'content': user_message_template.format(review=review_example1)},
        {'role':'assistant', 'content': f"{assistant_output_example1}"},
        {'role':'user', 'content': user_message_template.format(review=review_example2)},
        {'role':'assistant', 'content': f"{assistant_output_example2}"},
        {'role':'user', 'content': user_message_template.format(review=review_example3)},
        {'role':'assistant', 'content': f"{assistant_output_example3}"},
        {'role':'user', 'content': user_message_template.format(review=review_example4)},
        {'role':'assistant', 'content': f"{assistant_output_example4}"}
                        ]

    few_shot_examples_str = json.dumps(few_shot_examples)


    sample_reviews = sample_reviews_df.review_text.values
    sentiment_predictions = []
    # generate_llama_response( few_shot_system_message , few_shot_examples_str  , input_text , 0.1 )
    for sample_review in tqdm(sample_reviews):
        try:
            sentiment_predictions.append(generate_llama_response( few_shot_system_message , few_shot_examples_str , sample_review , 0.1 ))
        except Exception as e:
            print(e)
            sentiment_predictions.append("")
    sentiment_predictions
    sample_reviews_df["sentiment_prediction"] = sentiment_predictions

    pattern = r'\[\s*.*?\s*:\s*\{.*?\}\s*\]'

    # Function to extract the sentiment data
    def extract_sentiment(text):
        match = re.findall(pattern, text)
        return match[0] if match else None


    # Apply the function to each row in the DataFrame
    sample_reviews_df['extracted_sentiment'] = sample_reviews_df['sentiment_prediction'].apply(extract_sentiment)

    sample_reviews_df
    # Function to extract product type
    print(sample_reviews_df['extracted_sentiment'])

    # Apply the functions to each row in the DataFrame
    try:
          sample_reviews_df['predicted_product_type'] = sample_reviews_df['extracted_sentiment'].apply(extract_product_type)
          sample_reviews_df['predicted_aspect_based_sentiment'] = sample_reviews_df['extracted_sentiment'].apply(extract_aspect_based_sentiment)
    except Exception as e:
            print(e)
            print(sample_reviews_df['predicted_product_type'] )

    print(sample_reviews_examples_df['sentiment_prediction'])
    sample_reviews_df.predicted_aspect_based_sentiment.value_counts()



    # (B) Compute Accuracy Here
    combined_accuracy = "__________"
    # compute_combined_aspect_and_product_accuracy(sample_reviews_df)

    print(f"Combined Product Type and Aspect-Based Sentiment Accuracy: {combined_accuracy}%")
    res = combined_accuracy
    accuracy_list.append(res)

In [ ]:
accuracy_list

In [ ]:
mean_accuracy = "__________"

In [ ]:
mean_accuracy

###*Evaluation Few Shot*

In [ ]:
gold_examples

In [ ]:
def evaluate_prompt(prompt, gold_examples, user_message_template):

    """
    Return the micro-F1 score for predictions on gold examples.
    For each example, we make a prediction using the prompt. Gold labels and
    model predictions are aggregated into lists and compared to compute the
    F1 score.

    Args:
        prompt (List): list of messages in the Open AI prompt format
        gold_examples (str): JSON string with list of gold examples
        user_message_template (str): string with a placeholder for movie reviews

    Output:
        micro_f1_score (float): Micro-F1 score computed by comparing model predictions
                                with ground truth
    """

    model_predictions, ground_truths = [], []

    for example in json.loads(gold_examples):
        gold_input = example['review_text']
        user_input = [
            {
               user_message_template.format(review=gold_input)
            }
        ]

        try:
            prediction = generate_llama_response( few_shot_system_message , few_shot_examples_str  , user_input , 0.1 )

            print("prediction : " + prediction + "\n")

            prediction_extracted = extract_sentiment(prediction)
            predicted_product_type = extract_product_type(prediction_extracted)
            predicted_aspect_based_sentiment = extract_aspect_based_sentiment(prediction_extracted)

            final_prediction = predicted_product_type + ":" + predicted_aspect_based_sentiment

            # sentiment = match.group().lower() if match else "Sentiment not found."
            print("model_prediction : " + final_prediction.lower() + "\n")

            model_predictions.append(final_prediction.lower()) # <- removes extraneous white space and lowercases output
            ground_truths.append( example['product_type'].lower() + ":" + example['aspects_review'].lower() )

            print("ground truth : " + example['product_type'].lower() + ":" + example['aspects_review'].lower() + "\n")

        except Exception as e:
            continue

    micro_f1_score = f1_score(ground_truths, model_predictions, average="micro")

    return micro_f1_score

###**(C) Evaluate the Results**

In [ ]:
"__________"

#*Fine-tuning the Llama 2 Model for Summarization*

##**Question 10: Dataset Preprocessing for Text Summarization (5 Marks)**

(A) Read the Dataset (2 Marks)

(B) Split the Datset (3 Marks)

###**(A) Load Dataset**

In [ ]:
sample_reviews_df = "__________"

In [ ]:
sample_reviews_df['dialogue'] = 'customer : ' + sample_reviews_df['review_text'] + '\n' + 'response : ' + sample_reviews_df['response'] + '\n'

In [ ]:
sample_reviews_df['id'] = sample_reviews_df['customer_id']

In [ ]:
sample_reviews_df = sample_reviews_df[['id', 'review_sentiment' ,'dialogue','summary']]

###**(B) Split Dataset**

In [ ]:
# Separate positive and negative reviews
positive_reviews = "__________"
negative_reviews = "__________"

# Sample 2 positive and 2 negative reviews for gold examples
positive_gold_examples = positive_reviews.sample(2, random_state=40)
negative_gold_examples = negative_reviews.sample(2, random_state=40)

# Concatenate positive and negative gold examples
sample_reviews_gold_examples_df = "__________"

# Create the training set by excluding gold examples
sample_reviews_examples_df = "__________"

# Print the shapes of the datasets
print("Training Set Shape:", sample_reviews_examples_df.shape)
print("Gold Examples Shape:", sample_reviews_gold_examples_df.shape)


##*Training*

##**Question 11: Model Setup for Fine-tuning (5 Marks)**

(A) Load Model Name (1 Marks)

(B) Create Examples (2 Marks)

(C) Initialize Model (2 Marks)

###**(A) Load llama-2-7b-bnb-4bit model Name**

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="___________",
    max_seq_length=2048,
    dtype=None, # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
    load_in_4bit=True # Use 4bit quantization to reduce memory usage.
)

In [ ]:
model

In [ ]:
tokenizer

In [ ]:
EOS_TOKEN = tokenizer.eos_token

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0, # Supports any, but = 0 is optimized
    bias="none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing=True,
    random_state=42,
    use_rslora=False,
    loftq_config=None
)

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [ ]:
def create_examples_with_seed(dataset, n=2, random_seed=None):
    """
    Return two DataFrames with randomized examples of size 2n with two classes.
    Create subsets of each class, choose random samples from the subsets,
    merge and randomize the order of samples in the merged list.
    Each run of this function creates a different random sample of examples
    chosen from the training data.

    Args:
        dataset (DataFrame): A DataFrame with examples (text + label)
        n (int): number of examples of each class to be selected
        random_seed (int): seed for reproducibility (default is None)

    Output:
        few_shot_examples_df (DataFrame): A DataFrame with examples in random order
        new_df (DataFrame): A new DataFrame excluding selected examples
    """

    positive_reviews = (dataset.review_sentiment == 'Positive')
    negative_reviews = (dataset.review_sentiment == 'Negative')
    columns_to_select = ['id', 'review_sentiment' ,'dialogue','summary']

    # Set a fixed random seed for reproducibility
    np.random.seed(random_seed)

    positive_examples = dataset.loc[positive_reviews, columns_to_select].sample(n)
    negative_examples = dataset.loc[negative_reviews, columns_to_select].sample(n)

    few_shot_examples_df = pd.concat([positive_examples, negative_examples])
    # sampling without replacement is equivalent to random shuffling
    few_shot_examples_df = few_shot_examples_df.sample(2 * n, replace=False)

    # Create a new DataFrame excluding selected examples
    new_df = dataset.drop(index=few_shot_examples_df.index)

    return few_shot_examples_df, new_df


###**(B) Create Examples by mentioning the seed value used earlier and n=2**

In [ ]:
sample_reviews_train_examples_df , sample_reviews_validation_examples_df = "__________"

In [ ]:
training_dataset = datasets.Dataset.from_pandas(sample_reviews_train_examples_df)
validation_dataset = datasets.Dataset.from_pandas(sample_reviews_validation_examples_df)


In [ ]:
def prompt_formatter(example, prompt_template):
    instruction='Summarize the following dialogue'
    dialogue=example["dialogue"]
    summary=example["summary"]

    formatted_prompt = prompt_template.format(instruction, dialogue, summary)

    return {'formatted_prompt': formatted_prompt}

In [ ]:
formatted_training_dataset = training_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

In [ ]:
formatted_validation_dataset = validation_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

###**(C) Initialize Model Parameters**

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    # Input formatted training dataset
    train_dataset="__________",
    # Input formatted validation dataset
    eval_dataset="__________",
    dataset_text_field = "formatted_prompt",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False, # Increases efficiency for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs"
    )
)

##*Start of Training*

##**Question 12: Train and Save the Model (5 Marks)**

(A) Train the Model (3 Marks)

(B) Save the Model (2 Marks)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

###**(A) Train Model**

In [ ]:
# Start Training
trainer_stats = "__________"

In [ ]:
trainer_stats

###*Inference*

In [ ]:
test_dataset = datasets.Dataset.from_pandas(sample_reviews_gold_examples_df)

In [ ]:
test_dataset[0]

In [ ]:
instruction = "Summarize the following dialogue"
test_dialogue = test_dataset[0]['dialogue']
test_summary = test_dataset[0]['summary']

In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        instruction,
        test_dialogue,
        "", # leave output blank for generation
    )
], return_tensors="pt").to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True,
    do_sample=True,
    temperature=0.2
)

In [ ]:
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
test_summary

The following code allows us to run shell commands on Colab (we need shell commands to check file sizes of the estimated models).

In [ ]:
import locale

In [ ]:
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

In [ ]:
lora_model_name = "dialogue-summarizer-llama"

###**(B) Save Model**

In [ ]:
# save the model using save_pretrained function from model
"__________"

In [ ]:
!ls -lh {lora_model_name}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r {lora_model_name} /content/drive/MyDrive/

##**Question 13: Evaluate Model Performance (10 Marks)**

(A) Load Base Model (2 Marks)

(B) Evaluate Performance of Base Model (3 Marks)

(C) Load Trained Model (2 Marks)

(D) Evaluate Performance of Trained Model (3 Marks)

##*Llama 2 Base Model Performance*

###**(A) Load Base Model**

In [ ]:
baseline_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="__________",
    max_seq_length=2048,
    dtype=None, # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
    load_in_4bit=True # Use 4bit quantization to reduce memory usage.
)

In [ ]:
FastLanguageModel.for_inference(baseline_model)

In [ ]:
test_dataset = datasets.Dataset.from_pandas(sample_reviews_gold_examples_df)

###*Single Inference*

In [ ]:
torch.cuda.empty_cache()

In [ ]:
instruction = "Summarize the following dialogue"
test_dialogue = test_dataset[0]['dialogue']
test_summary = test_dataset[0]['summary']

In [ ]:
input = tokenizer(
    alpaca_prompt.format(
        instruction,
        test_dialogue,
        ""
    ), return_tensors="pt"
).to("cuda")

In [ ]:
output = baseline_model.generate(
    **input,
    max_new_tokens=128,
    use_cache=True,
    do_sample=True,
    temperature=0.2
)

In [ ]:
print(tokenizer.decode(output[0]))

###*Batch Inference*

In [ ]:
torch.cuda.empty_cache()

In [ ]:
instruction = "Summarize the following dialogue"

In [ ]:
test_dialogues = [sample['dialogue'] for sample in test_dataset]
test_summaries = [sample['summary'] for sample in test_dataset]

In [ ]:
def extract_summary_from_string(input_string):
    try:
        # Assuming the response is between ### Response: and </s>
        summary_start = input_string.rfind('### Response:\n') + 14 # number of characters in '### Response:\n'
        summary_end = input_string.rfind('</s>')
        summary_str = input_string[summary_start:summary_end]

        return summary_str
    except Exception as e:
        print(f"Error decoding string: {e}")
        return None

In [ ]:
predicted_summaries = []

In [ ]:
for sample_dialogue in test_dialogues:
  input = tokenizer(
    alpaca_prompt.format(
        instruction,
        sample_dialogue,
        ""
    ), return_tensors="pt"
  ).to("cuda")

  outputs = baseline_model.generate(
    **input,
    max_new_tokens=256,
    use_cache=True
  )

  predicted_summary = tokenizer.decode(outputs[0])

  output_str = extract_summary_from_string(predicted_summary)

  predicted_summaries.append(output_str)

###*Evaluate*

###**(B) Evaluate Performance of Base Model**

In [ ]:
bert_scorer = evaluate.load("bertscore")

In [ ]:
# Provide Prediction Summaries and Test Summaries as input
score = bert_scorer.compute(
    predictions="_________",
    references="_________",
    lang="en",
    rescale_with_baseline=True
)

In [ ]:
#  Calculate Average Bert Score . Average Bert Score is sum of f1 score divided by number of samples
"__________"

###*Llama 2 Trained (Fine-tuned) Model Summarizer*

In [ ]:
lora_model_name = "dialogue-summarizer-llama"

In [ ]:
drive.mount('/content/drive')

In [ ]:
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

In [ ]:
!cp -r /content/drive/MyDrive/{lora_model_name} .

###**(C) Load Trained Model**

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="__________",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True
)

In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
test_dataset = datasets.Dataset.from_pandas(sample_reviews_gold_examples_df)

###*Single Inference*

In [ ]:
torch.cuda.empty_cache()

In [ ]:
instruction = "Summarize the following dialogue"
test_dialogue = test_dataset[0]['dialogue']
test_summary = test_dataset[0]['summary']

In [ ]:
input = tokenizer(
    alpaca_prompt.format(
        instruction,
        test_dialogue,
        ""
    ), return_tensors="pt"
).to("cuda")

In [ ]:
output = model.generate(
    **input,
    max_new_tokens=128,
    use_cache=True,
    do_sample=True,
    temperature=0.2
)

In [ ]:
print(tokenizer.decode(output[0]))

###*Batch Inference*

In [ ]:
torch.cuda.empty_cache()

In [ ]:
instruction = "Summarize the following dialogue"

In [ ]:
# test_size = 4

In [ ]:
test_dialogues = [sample['dialogue'] for sample in test_dataset]
test_summaries = [sample['summary'] for sample in test_dataset]

In [ ]:
def extract_summary_from_string(input_string):
    try:
        # Assuming the response is between ### Response: and </s>
        summary_start = input_string.rfind('### Response:\n') + 14 # number of characters in '### Response:\n'
        summary_end = input_string.rfind('</s>')
        summary_str = input_string[summary_start:summary_end]

        return summary_str
    except Exception as e:
        print(f"Error decoding string: {e}")
        return None

In [ ]:
predicted_summaries = []

In [ ]:
for sample_dialogue in test_dialogues:
  input = tokenizer(
    alpaca_prompt.format(
        instruction,
        sample_dialogue,
        ""
    ), return_tensors="pt"
  ).to("cuda")

  outputs = model.generate(
    **input,
    max_new_tokens=256,
    use_cache=True
  )

  predicted_summary = tokenizer.decode(outputs[0])

  output_str = extract_summary_from_string(predicted_summary)

  predicted_summaries.append(output_str)

###*Evaluate*

###**(D) Evaluate Performance of Trained Model**

In [ ]:
# Input prediction summaries and test summaries in bert scorer
score = bert_scorer.compute(
    predictions="__________",
    references="__________",
    lang="en",
    rescale_with_baseline=True
)

In [ ]:
#  Calculate Average Bert Score . Average Bert Score is sum of f1 score divided by number of samples
"__________"

##**Question 14: Observations, Insights and Business Recommendations (20 Marks)**